# JRC Global River Flood Hazard Maps v2.1


In [ ]:
# Site configuration — uses transformation/flood_hazard city configs
import os
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_FLOOD_HAZARD = None
for _candidate in [_HERE, *_HERE.parents]:
    _probe = _candidate / "flood_hazard" if _candidate.name != "flood_hazard" else _candidate
    if (_probe / "site_config.py").is_file() and (_probe / "config" / "sites").is_dir():
        _FLOOD_HAZARD = _probe
        break
if _FLOOD_HAZARD is None:
    raise FileNotFoundError("Could not locate transformation/flood_hazard from notebook cwd")

sys.path.insert(0, str(_FLOOD_HAZARD))
from site_config import configured_path, load_site_config

FLOOD_HAZARD_ROOT = _FLOOD_HAZARD
FLOODS_ROOT = FLOOD_HAZARD_ROOT  # backward-compatible alias

# Set the city here (edit this). Optional: export FLOODS_SITE=... to override.
SITE_SLUG = "plymouth"  # porto_alegre | plymouth | edina | richfield | rochester | apple_valley
if "FLOODS_SITE" in os.environ:
    SITE_SLUG = os.environ["FLOODS_SITE"]
SITE_CONFIG = load_site_config(SITE_SLUG, FLOOD_HAZARD_ROOT)
SITE_ROOT = SITE_CONFIG["paths_abs"]["site_root"]
INPUT_DIR = SITE_CONFIG["paths_abs"]["data_input"]
INTERMEDIATE_DIR = SITE_CONFIG["paths_abs"]["data_intermediate"]
OUTPUT_DIR = SITE_CONFIG["paths_abs"]["data_output"]
OUT_ROOT = SITE_CONFIG["paths_abs"]["out"]
CACHE_DIR = SITE_CONFIG["paths_abs"]["cache"]
STYLES_DIR = SITE_CONFIG["paths_abs"]["styles"]
OUTPUT_PREFIX = SITE_CONFIG["output_prefix"]
print(f"Flood hazard site: {SITE_CONFIG['display_name']} ({SITE_SLUG})")
print(f"Config: {SITE_CONFIG['config_path']}")
print(f"Inputs -> {INPUT_DIR}")


Dataset Availability
- 2024-03-16T00:00:00Z–2024-03-16T23:59:59Z

Pixel size
- 90 meters (all bands)

Dataset Producer
- Joint Research Centre

Terms of Use
- The JRC datasets are available without restriction on use or distribution.


## Methodology

### Dataset context and interpretation

**JRC Global River Flood Hazard Maps Version 2.1** is conceptually different from Aqueduct Floods and the Global Flood Database:

- It is **not** a historical time series of observed events.
- It does **not** include future climate scenarios.
- It is **not** filtered by event date.
- It is a **static modeled flood-depth product** for multiple return periods.

The dataset represents riverine flooding along the global river network for return periods from **1-in-10 years to 1-in-500 years**, produced using LISFLOOD discharges and LISFLOOD-FP inundation simulations.

### Normalization methodology (impact-oriented)

The flood hazard score is derived from JRC/CEMS GLOFAS Flood Hazard v2.1 RP100 inundation depth. Depth values are normalized using an impact-based threshold where depths of 2 m or greater are assigned the maximum hazard score of 1. This preserves comparability across cities and avoids compressing urban-relevant flood depths using global maximum values.

#### Why this normalization

- It is easier to communicate to non-technical stakeholders (thresholds in meters).
- It supports risk-screening decisions by mapping depth ranges to impact-relevant severity classes.
- It provides a stable 0-1 score that is directly comparable with other normalized hazard layers in the composite workflow.

#### How normalization is applied

Given flood depth `d` (meters), define:

- `0 <= d <= 0.15` m -> `hazard_score = 0.00`
- `0.15 < d <= 0.5` m -> `hazard_score = 0.25`
- `0.5 < d <= 1.0` m -> `hazard_score = 0.50`
- `1.0 < d <= 2.0` m -> `hazard_score = 0.75`
- `d > 2.0` m -> `hazard_score = 1.00`

#### Methodological note

This score should be interpreted as a **hazard severity class from modeled fluvial depth**, not as a full flood-risk metric (it does not include exposure and vulnerability).


In [2]:
import ee
import geemap

In [3]:
# 1) Init EE
ee.Authenticate()
ee.Initialize(
    project="eecc-maureen",
    opt_url="https://earthengine-highvolume.googleapis.com"
)


In [ ]:
# 2) Site ROI: use the site polygon when available; fall back to bbox.
import json


def load_site_roi() -> ee.Geometry:
    boundary_path = SITE_CONFIG["boundary_path_abs"]
    if boundary_path.exists():
        data = json.loads(boundary_path.read_text())
        if data.get("type") == "FeatureCollection":
            features = [
                ee.Feature(ee.Geometry(feature["geometry"]), feature.get("properties", {}))
                for feature in data.get("features", [])
                if feature.get("geometry")
            ]
            if features:
                return ee.FeatureCollection(features).geometry()
        if data.get("type") == "Feature":
            return ee.Geometry(data["geometry"])
        if data.get("type") in {"Polygon", "MultiPolygon", "GeometryCollection"}:
            return ee.Geometry(data)

    return ee.Geometry.Rectangle(SITE_CONFIG["bbox"])


roi = load_site_roi()
print(f"ROI loaded for {SITE_CONFIG['display_name']} from {SITE_CONFIG['boundary_path_abs']}")


In [ ]:
# Load JRC flood hazard and extract RP100 depth (raw meters)
dataset = ee.ImageCollection("JRC/CEMS_GLOFAS/FloodHazard/v2_1")
image = dataset.mosaic()

depth100 = image.select("RP100_depth").clip(roi).rename("depth_rp100_m")
print("Raw depth layer ready:", depth100.bandNames().getInfo())


### Raw depth distribution (before normalization)

Within-city ranking: inspect **raw inundation depth (m)** first.
If almost all mass sits below 0.15 m, global impact classes will collapse to ~0.


In [ ]:
# Histogram of RAW depth (meters) before impact-class normalization
import numpy as np
import matplotlib.pyplot as plt

_stats = depth100.reduceRegion(
    reducer=ee.Reducer.minMax()
        .combine(ee.Reducer.percentile([10, 25, 50, 75, 90, 95, 99]), sharedInputs=True)
        .combine(ee.Reducer.count(), sharedInputs=True),
    geometry=roi,
    scale=90,
    maxPixels=1e9,
    bestEffort=True,
).getInfo()
print("Raw depth stats (JRC RP100):")
for k, v in sorted(_stats.items()):
    print(f"  {k}: {v}")

_sample = depth100.sample(region=roi, scale=90, numPixels=4000, seed=42, geometries=False)
_vals = np.array(_sample.aggregate_array("depth_rp100_m").getInfo(), dtype=float)
_vals = _vals[np.isfinite(_vals)]
print(f"Sampled pixels: {_vals.size}")
if _vals.size:
    print(
        f"share <=0.15 m: {(_vals <= 0.15).mean()*100:.1f}% | "
        f"<=0.5 m: {(_vals <= 0.5).mean()*100:.1f}% | "
        f"<=1 m: {(_vals <= 1).mean()*100:.1f}% | "
        f">2 m: {(_vals > 2).mean()*100:.1f}%"
    )
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(_vals, bins=40, color="steelblue", edgecolor="none")
    for x, lab in [(0.15, "0.15"), (0.5, "0.5"), (1.0, "1.0"), (2.0, "2.0")]:
        ax.axvline(x, color="crimson", linestyle="--", linewidth=1, label=f"class {lab} m")
    ax.set_xlabel("Inundation depth (m)")
    ax.set_ylabel("Pixel count (sample)")
    ax.set_title(f"Raw depth — {SITE_CONFIG['display_name']} (JRC RP100)")
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No valid depth samples in AOI — check product coverage / ROI.")


In [ ]:
# Impact-class normalization (global thresholds)
# Review the raw-depth histogram above before trusting this for within-city ranking.
hazard_score_rp100 = depth100.expression(
    "(d <= 0.15) ? 0"
    ": (d <= 0.5) ? 0.25"
    ": (d <= 1.0) ? 0.5"
    ": (d <= 2.0) ? 0.75"
    ": 1",
    {"d": depth100},
).rename("hazard_score_rp100")

hazard_score_rp100 = hazard_score_rp100.updateMask(depth100.mask()).clip(roi)
print("Layers ready: depth100, hazard_score_rp100")


In [6]:
# 5) Visualization
m = geemap.Map()
m.centerObject(roi, 11)

depth_vis = {
    "min": 0,
    "max": 3,
    "palette": ["f7fbff", "c6dbef", "6baed6", "2171b5", "08306b"]
}

hazard_vis = {
    "min": 0,
    "max": 1,
    "palette": ["f7f7f7", "c7e9c0", "74c476", "238b45", "00441b"]
}

m.addLayer(depth100, depth_vis, "JRC RP100 depth (m)")
m.addLayer(hazard_score_rp100, hazard_vis, "JRC RP100 hazard score (impact classes)")
m.addLayer(ee.Image().paint(roi, 1, 2), {"palette": ["red"]}, "ROI boundary")

m.add_legend(
    title="Hazard score classes",
    keys=[
        "0.00 -> 0-0.15 m",
        "0.25 -> 0.15-0.5 m",
        "0.50 -> 0.5-1.0 m",
        "0.75 -> 1.0-2.0 m",
        "1.00 -> >2.0 m",
    ],
    colors=["#f7f7f7", "#c7e9c0", "#74c476", "#238b45", "#00441b"],
    position="bottomright",
)

m

Map(center=[46.453656143116, -93.36500000000001], controls=(WidgetControl(options=['position', 'transparent_bg…

In [ ]:
# --- Export JRC RP100 depth + normalized hazard to site input/ (local by default) ---
# Requires: roi, depth100, hazard_score_rp100, SITE_CONFIG, INPUT_DIR, OUTPUT_PREFIX
# Override: export GEE_EXPORT_MODE=drive  (legacy Google Drive)

from gee_local_export import export_image_to_input

scale_m = 90
crs = "EPSG:4326"
drive_folder = "OEF_JRC_FloodHazard"

export_image_to_input(
    depth100.toFloat().clip(roi),
    filename=SITE_CONFIG["layers"]["jrc_depth"],
    region=roi,
    scale=scale_m,
    input_dir=INPUT_DIR,
    crs=crs,
    description=f"jrc_rp100_depth_{OUTPUT_PREFIX}",
    drive_folder=drive_folder,
)
export_image_to_input(
    hazard_score_rp100.toFloat().clip(roi),
    filename=SITE_CONFIG["layers"]["jrc_norm"],
    region=roi,
    scale=scale_m,
    input_dir=INPUT_DIR,
    crs=crs,
    description=f"jrc_rp100_depth_norm_{OUTPUT_PREFIX}",
    drive_folder=drive_folder,
)

print("Exports complete (see paths above). Files land under:", INPUT_DIR)


#### Step 6. Convert RP100 Depth to COG and Generate Tiles

Publish the local `sites/<site_slug>/data/input/jrc_rp100_depth_poa.tif` raster for web maps: COG + colorized XYZ tiles + value-encoded XYZ tiles for hover lookup.

Requires GDAL CLI (`gdal_translate`, `gdaldem`, `gdal_calc.py`, `gdal2tiles.py`) and `data/jrc_rp100_depth_colors.txt`.

In [ ]:
# Convert JRC RP100 depth GeoTIFF to COG + visual tiles + value tiles
# Input expected after downloading the Earth Engine export from Google Drive.
from pathlib import Path
import shutil
import subprocess


PROJECT_ROOT = FLOODS_ROOT
in_tif = INPUT_DIR / SITE_CONFIG["layers"]["jrc_depth"]
out_dir = OUT_ROOT / "jrc_rp100_depth"
colors_txt = STYLES_DIR / "jrc_rp100_depth_colors.txt"

slug = "jrc_rp100_depth"
cog_tif = out_dir / f"{slug}_cog.tif"
colorized_tif = out_dir / f"{slug}_colorized.tif"
value_encoded_tif = out_dir / f"{slug}_value_encoded_rgb.tif"
tiles_dir = out_dir / "tiles_visual"
value_tiles_dir = out_dir / "tiles_values"
decode_txt = out_dir / f"{slug}_value_tiles_decode.txt"

out_dir.mkdir(parents=True, exist_ok=True)
if not in_tif.exists():
    raise FileNotFoundError(f"Missing input raster: {in_tif}")
if not colors_txt.exists():
    raise FileNotFoundError(f"Missing color table: {colors_txt}")

print("Input:", in_tif)
print("Output dir:", out_dir)

# 1) COG preserving raw depth in meters.
subprocess.run([
    "gdal_translate", str(in_tif), str(cog_tif),
    "-of", "COG",
    "-ot", "Float32",
    "-co", "COMPRESS=DEFLATE",
    "-co", "RESAMPLING=NEAREST",
    "-co", "OVERVIEWS=AUTO",
], check=True)
print("Created COG:", cog_tif)

# 2) Colorized raster + visual XYZ tiles.
subprocess.run([
    "gdaldem", "color-relief",
    str(cog_tif), str(colors_txt), str(colorized_tif),
    "-alpha",
], check=True)
print("Created colorized raster:", colorized_tif)

# gdal2tiles preflight: use the Python interpreter referenced by gdal2tiles.py.
gdal2tiles = shutil.which("gdal2tiles.py")
if not gdal2tiles:
    raise RuntimeError("gdal2tiles.py not found in PATH. Install GDAL (e.g. brew install gdal).")

gdal2tiles_python = None
with open(gdal2tiles, "r", encoding="utf-8", errors="ignore") as f:
    first_line = f.readline().strip()
if first_line.startswith("#!"):
    shebang_parts = first_line[2:].split()
    if shebang_parts:
        if shebang_parts[0].endswith("env") and len(shebang_parts) > 1:
            gdal2tiles_python = shutil.which(shebang_parts[1])
        else:
            gdal2tiles_python = shebang_parts[0]
if not gdal2tiles_python:
    gdal2tiles_python = shutil.which("python3") or shutil.which("python")
subprocess.run([gdal2tiles_python, "-c", "import numpy"], check=True, capture_output=True)

tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py",
    "-r", "near",
    "-z", "8-15",
    "--xyz",
    "-w", "none",
    str(colorized_tif),
    str(tiles_dir),
], check=True)
print("Visual tiles:", tiles_dir)

# 3) Value tiles for client-side hover.
# Encodes depth in millimeters plus 1 so encoded RGB value 0 can mean nodata.
# Decode: encoded = R + 256*G + 65536*B; depth_m = (encoded - 1) / 1000; encoded == 0 => nodata.
base_expr = (
    "numpy.where(numpy.isnan(A), 0, "
    "numpy.rint(numpy.clip(A,0,10000)*1000).astype(numpy.int64) + 1)"
)
subprocess.run([
    "gdal_calc.py",
    "-A", str(cog_tif),
    "--calc", f"bitwise_and({base_expr},255)",
    "--calc", f"bitwise_and(right_shift({base_expr},8),255)",
    "--calc", f"bitwise_and(right_shift({base_expr},16),255)",
    "--type", "Byte",
    "--NoDataValue", "0",
    "--overwrite",
    "--outfile", str(value_encoded_tif),
], check=True)

value_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py",
    "-r", "near",
    "-z", "8-15",
    "--xyz",
    "-w", "none",
    str(value_encoded_tif),
    str(value_tiles_dir),
], check=True)

metadata = """JRC RP100 depth value tiles

Source raster: sites/<site_slug>/data/input/jrc_rp100_depth_poa.tif
COG: sites/<site_slug>/out/jrc_rp100_depth/jrc_rp100_depth_cog.tif
Visual tiles: sites/<site_slug>/out/jrc_rp100_depth/tiles_visual/{z}/{x}/{y}.png
Value tiles: sites/<site_slug>/out/jrc_rp100_depth/tiles_values/{z}/{x}/{y}.png

Value tile encoding:
encoded = R + 256 * G + 65536 * B
if encoded == 0: value is nodata
else: depth_m = (encoded - 1) / 1000

Depth values are stored in meters and encoded at millimeter precision.
"""
decode_txt.write_text(metadata, encoding="utf-8")

print("Value tiles:", value_tiles_dir)
print("Decode metadata:", decode_txt)
print("Decode: encoded = R + 256*G + 65536*B; depth_m = (encoded - 1) / 1000; encoded 0 = nodata")
